# Modelling Viewcount growth of top 500 movies post 2000

Inspired by the Paper "Bio-Inspired Models for Characterizing YouTube Viewcount", a review of a movie is counted as a view. The top 500 most reviewed movies post 2000 are then compared with two bio isnpired viewcount models and classfied into "viral" or "non-viral" movies. The models used for this calssification are the "Gompertz-Model" and the "negative exponetial modell". Both assume a fixed target population, which seams fitting for a movie release. ANd the negative exponential model tries to model the popularity gained via braodcasted advertisement, which also is pretty fitting

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
movies_dtype = {
    "movieId": "int32",
    "title": "string",
    "genres": "string"
}
ratings_dtype = {
    "userId": "int32",
    "movieId": "int32",
    "rating": "float32",
    "timestamp": "int64"
}
movies_df = pd.read_csv("./../data/ml-latest/movies.csv", dtype=movies_dtype)
ratings_df = pd.read_csv("./../data/ml-latest/ratings.csv", dtype=ratings_dtype)


# Fill in release year based on regex from title as col
movies_df["year"] = movies_df["title"].str.extract(r"\((\d{4})\)").astype("Int64")

post_2000_movies = movies_df[movies_df["year"] >= 2000].copy()
post_2000_ids = post_2000_movies["movieId"]

review_counts = ratings_df["movieId"].value_counts()
post_2000_review_counts = review_counts[review_counts.index.isin(post_2000_ids)]

top_500_movies = post_2000_review_counts.nlargest(500)
top_500_ids = top_500_movies.index.values

movies_df = post_2000_movies[post_2000_movies["movieId"].isin(top_500_ids)]
ratings_df = ratings_df[ratings_df["movieId"].isin(top_500_ids)]

In [ ]:
cum_count_time_series = {}

for movie_id in movies_df["movieId"]:
    movie_ratings = ratings_df[ratings_df["movieId"] == movie_id].sort_values("timestamp")

    # if len(movie_ratings) < 400:
    #     continue

    timestamps = movie_ratings["timestamp"].values

    t = timestamps - timestamps[0]

    total_time = t[-1]

    cum = np.arange(1, len(movie_ratings) + 1, dtype=float)
    cum_max = cum[-1]

    t_norm = t / total_time
    cum_norm = cum / cum_max

    cum_count_time_series[movie_id] = {
        "title": movies_df.loc[movies_df["movieId"] == movie_id, "title"].values[0],
        "t": t_norm,
        "y": cum_norm,
        "total_reviews": cum_max,
        "avg_rating": movie_ratings["rating"].mean()
    }

In [ ]:
def neg_exponential(t, M, lam, S0=0):
    return S0 + (M - S0) * (1.0 - np.exp(-lam * t))

def gompertz(t, M, lam, S0=0):
    return M * np.exp(-np.log(M / S0) * np.exp(-lam * t))

In [ ]:
from scipy.optimize import curve_fit

results = []

for movie_id, data in cum_count_time_series.items():
    t_vals = data["t"]
    y_vals = data["y"]

    S0 = y_vals[0]
    M=1.0           # As total count is staic and normalized

    def fit_neg(t, lam):
        return neg_exponential(t, M, lam, S0)

    def fit_gomp(t, lam):
        return gompertz(t, M, lam, S0)

    popt_exp, _ = curve_fit(fit_neg, t_vals, y_vals, bounds=(0, np.inf))
    lam_exp = popt_exp[0]
    mse_exp = np.mean((fit_neg(t_vals, lam_exp) - y_vals) ** 2)
    
    popt_gomp, _ = curve_fit(fit_gomp, t_vals, y_vals, bounds=(0, np.inf))
    lam_gomp = popt_gomp[0]
    mse_gomp = np.mean((fit_gomp(t_vals, lam_gomp) - y_vals) ** 2)

    best_model = "Negative Exponential (Broadcast)" if mse_exp <= mse_gomp else "Gompertz (Viral S-Curve)"
    best_mse = min(mse_exp, mse_gomp)
    
    results.append({
        "movieId": movie_id,
        "title": data["title"],
        "total_reviews": data["total_reviews"],
        "avg_rating": data["avg_rating"],
        "best_model": best_model,
        "mse": best_mse
    })

df_results = pd.DataFrame(results)

model_counts = df_results["best_model"].value_counts()
print("Model Distribution across Top 100 Movies:")
print(model_counts)